In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os

def build_densenet121(input_shape=(224, 224, 3), num_classes=4):
    # 1. Load Pre-trained DenseNet121 base (exclude top classification layer)
    # Using 'imagenet' weights helps the model recognize basic patterns/shapes
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze the base model to prevent losing pre-trained features during initial training
    base_model.trainable = False

    # 2. Create the Custom Head for Pneumonia Classification
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(), # Reduces spatial dimensions
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.4),            # Stronger dropout to prevent overfitting
        layers.Dense(num_classes, activation='softmax') # 4 units for your 4 classes
    ])

    return model

# --- Configuration & Paths ---
# Use the directory you created in your Train_Test_Split_CNN_Data
BASE_DIR = "/content/Train_Test_Split_CNN_Data"
CHECKPOINT_PATH = "/content/drive/MyDrive/DenseNet_Checkpoints/densenet121_best.keras"

# 3. Load Datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "train"),
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "val"),
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

# 4. Initialize and Compile Model
model = build_densenet121()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Setup Callbacks (Safety for long training)
if not os.path.exists(os.path.dirname(CHECKPOINT_PATH)):
    os.makedirs(os.path.dirname(CHECKPOINT_PATH))

callbacks = [
    ModelCheckpoint(CHECKPOINT_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

# 6. Start Training
print("Starting DenseNet121 Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks
)